# 🗂️ Notebook 2: Key-Value Store — Data Model & APIs


## 🛠️ Setup

```bash
cd 06-system-designs/key-value-store
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Data model

Keep it minimal and byte-oriented. The store doesn't care what's inside the value.

```
key     : bytes (≤ 1 KB)     — short, unique identifier
value   : bytes (≤ 1 MB)     — opaque blob (JSON, protobuf, image, whatever)
version : monotonic counter  — used to resolve conflicts
```

**Why bytes?** The store is schema-less. The *application* encodes/decodes (JSON,
Avro, protobuf). The store only has to copy, index, and return bytes.


In [ ]:
from pydantic import BaseModel, Field

class KVRecord(BaseModel):
    key: str = Field(..., max_length=1024)
    value: str  # in practice: bytes; we use str for easy printing
    version: int = Field(..., ge=0)

r = KVRecord(key="user:42", value="alice", version=7)
print(r.model_dump_json(indent=2))

## HTTP API

```http
PUT    /kv/{key}   body: value         headers: X-Consistency: one|quorum|all
GET    /kv/{key}   → { value, version }
DELETE /kv/{key}
```

Internally, nodes gossip cluster membership and run **anti-entropy** (Merkle tree
diffs — Notebook 3) to reconcile replicas that drifted apart.


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class PutRequest(BaseModel):
    key: str = Field(..., max_length=1024)
    value: str
    consistency: Literal["one", "quorum", "all"] = "quorum"

class GetResponse(BaseModel):
    key: str
    value: str | None
    version: int | None
    replicas_answered: int

req = PutRequest(key="user:42", value="alice")
resp = GetResponse(key="user:42", value="alice", version=7, replicas_answered=2)
print("PUT  request :", req.model_dump_json())
print("GET  response:", resp.model_dump_json())

## 😱 Bad practice: storing raw values with no versioning

If the store keeps just `key → value` and nothing else, two concurrent writes
race and the loser is silently overwritten. Worse: we can't even *tell* a conflict
happened.


In [ ]:
# Single replica, no versioning — "last write wins" by wall clock order
store: dict[str, str] = {}

# Alice and Bob both read cart, add one item each, write back.
store["cart:42"] = "[apple]"          # initial

alice = store["cart:42"] + ",banana"  # Alice reads, appends
bob   = store["cart:42"] + ",cherry"  # Bob reads, appends
# Both write back, Bob arrives last:
store["cart:42"] = alice
store["cart:42"] = bob

print("final cart:", store["cart:42"])
print("➡ Alice's banana is LOST, silently. No warning, no log.")

## ✅ Better: version numbers — two different things people both call "LWW"

Once every record carries a version, two genuinely different strategies become possible, and
conflating them is a classic interview stumble:

| Strategy | On a conflicting write | Data loss? | Who does this |
|---|---|---|---|
| **Optimistic concurrency (CAS)** | reject it, tell the client to re-read and retry | none — the client is told | DynamoDB conditional writes, `etcd` compare-and-swap |
| **Last-write-wins (LWW)** | accept it, keep whichever timestamp is larger | **yes, silently** | Cassandra, DynamoDB global tables, S3 |

The first is safe but *not always available*: the client must be online to retry. The second is
always available but quietly throws away data. Let's run both.

In [ ]:
class CASStore:
    """Optimistic concurrency control: a write must state the version it read."""
    def __init__(self):
        self.data: dict[str, tuple[str, int]] = {}

    def put(self, key, value, expected_version):
        cur = self.data.get(key)
        cur_version = cur[1] if cur else 0
        if cur_version != expected_version:
            return f"rejected-conflict (you read v{expected_version}, store is at v{cur_version})"
        self.data[key] = (value, expected_version + 1)
        return f"written v{expected_version + 1}"

    def get(self, key):
        return self.data.get(key)

s = CASStore()
print(s.put("cart:42", "[apple]",        expected_version=0))
# Alice and Bob both read v1, both try to write v2:
print("alice:", s.put("cart:42", "[apple,banana]", expected_version=1))
print("bob  :", s.put("cart:42", "[apple,cherry]", expected_version=1))
print("stored:", s.get("cart:42"))
print("\n➡ Bob is REJECTED and knows it. He re-reads [apple,banana], re-applies his change,")
print("  and writes [apple,banana,cherry]. Nothing is lost — but Bob had to be around to retry.")


### 😱 …and now last-write-wins, silently losing a write

LWW never rejects anything, so a client always gets a `200 OK`. The resolution rule is just
"largest timestamp wins". That is fine when the timestamps are meaningful — and catastrophic
when they are not, because **wall clocks on different machines disagree**. NTP keeps servers
within tens of milliseconds on a good day, and unbounded apart on a bad one.

Below, Bob's node has a clock running 5 seconds *behind*. Bob writes strictly **after** Alice,
in real time, and both writes are acknowledged. Watch what the store ends up holding.

In [ ]:
class LWWStore:
    """Cassandra-style last-write-wins: highest timestamp wins, no write is ever refused."""
    def __init__(self):
        self.data: dict[str, tuple[str, float]] = {}

    def put(self, key, value, timestamp):
        cur = self.data.get(key)
        if cur is None or timestamp > cur[1]:
            self.data[key] = (value, timestamp)
        return "ok"          # <-- ALWAYS ok. The caller learns nothing.

    def get(self, key):
        return self.data.get(key)

# Two coordinator nodes. node_b's clock is 5 seconds slow — a very ordinary amount of skew.
CLOCK_SKEW = {"node_a": 0.0, "node_b": -5.0}
real_time = 1_000.0

lww = LWWStore()

# t=0   Alice writes through node_a
alice_real = real_time + 0
print(f"t={alice_real - real_time:>4.0f}s real  alice -> node_a  "
      f"(stamps it {alice_real + CLOCK_SKEW['node_a']:.0f})",
      lww.put("cart:42", "[apple,banana]", alice_real + CLOCK_SKEW["node_a"]))

# t=+2s Bob writes through node_b — LATER in real time, on a slow clock
bob_real = real_time + 2
print(f"t={bob_real - real_time:>4.0f}s real  bob   -> node_b  "
      f"(stamps it {bob_real + CLOCK_SKEW['node_b']:.0f})",
      lww.put("cart:42", "[apple,cherry]", bob_real + CLOCK_SKEW["node_b"]))

print(f"\nfinal value: {lww.get('cart:42')[0]}")
print("""
➡ Bob's write happened LAST and was acknowledged with a 200 OK — and it is gone.
  No error, no log line, no conflict counter. The only trace is a customer complaining
  that their cherry disappeared from the cart.

  This is not a bug in the implementation; it is the defined behaviour of LWW. The bug is
  choosing LWW for data where losing a write matters. LWW is correct for values that are
  genuinely "the latest reading wins" (a device's last-seen location, a cached price) and
  wrong for anything accumulative (carts, counters, sets, balances).

  Mitigations, in increasing order of cost:
    1. Generate the timestamp on ONE node (the coordinator), not per-client. Narrows the
       window; does not close it, because coordinators also drift.
    2. Use TrueTime/HLC-style bounded clocks (Spanner, CockroachDB) and pay for the
       hardware and the commit-wait latency.
    3. Stop resolving by time at all — track causality with vector clocks. Next section.""")


## 🌟 Best: vector clocks (preserve both sides of a conflict)

A vector clock is a dict `{node_id: counter}`. Every node increments its own
counter on write. On read, we compare clocks:

- `A` dominates `B`  →  A is strictly newer, keep A.
- Neither dominates  →  **concurrent writes** — return *both* as *siblings*
  and let the app merge (e.g., union the shopping carts).

This is how Dynamo and Riak avoid silent data loss.


In [ ]:
from dataclasses import dataclass, field

@dataclass
class Versioned:
    value: str
    clock: dict[str, int] = field(default_factory=dict)

def dominates(a: dict, b: dict) -> bool:
    """True if clock a is >= b on every node AND strictly greater somewhere."""
    all_ge = all(a.get(k, 0) >= v for k, v in b.items())
    any_gt = any(a.get(k, 0) >  b.get(k, 0) for k in set(a) | set(b))
    return all_ge and any_gt

def merge_siblings(values: list[Versioned]) -> list[Versioned]:
    """Keep only values whose clock is NOT dominated by another."""
    keep = []
    for v in values:
        if not any(dominates(w.clock, v.clock) for w in values if w is not v):
            keep.append(v)
    return keep

# Alice (node A) and Bob (node B) each write concurrently based on v1={A:1}
v1      = Versioned("[apple]",          clock={"A": 1})
alice   = Versioned("[apple,banana]",   clock={"A": 2})            # A advanced
bob     = Versioned("[apple,cherry]",   clock={"A": 1, "B": 1})    # B advanced

siblings = merge_siblings([alice, bob])
print(f"{len(siblings)} siblings returned to client:")
for s in siblings:
    print(" ", s)
print("➡ App merges → [apple, banana, cherry]. Nothing lost!")

## 🗄️ Storage layer — a glimpse of LSM-trees

Inside each node, how do we actually persist bytes on disk? Two classic options:

- **B-tree** (MySQL, Postgres) — great for reads, slower writes (random I/O).
- **LSM-tree** (Cassandra, RocksDB, LevelDB) — great for writes, reads need merging.

An LSM-tree writes:
1. **WAL** (write-ahead log): append the op to a file, fsync → durable.
2. **Memtable**: keep recent writes in a sorted in-memory structure.
3. When the memtable fills, **flush** it as an immutable sorted file (SSTable).
4. A background **compaction** merges small SSTables into larger ones.

KV stores are write-heavy, so LSM usually wins. Here's a toy version:


In [ ]:
import os, tempfile, json

class ToyLSM:
    def __init__(self, path):
        self.wal_path = path
        self.memtable: dict[str, str] = {}
        self.sstables: list[dict[str, str]] = []
        # Replay WAL (simulates crash recovery)
        if os.path.exists(path):
            with open(path) as f:
                for line in f:
                    op = json.loads(line)
                    self.memtable[op["k"]] = op["v"]

    def put(self, k, v):
        with open(self.wal_path, "a") as f:   # 1. append to WAL (durable)
            f.write(json.dumps({"k": k, "v": v}) + "\n")
        self.memtable[k] = v                  # 2. update memtable

    def get(self, k):
        if k in self.memtable:                # newest wins
            return self.memtable[k]
        for sst in reversed(self.sstables):   # search SSTables (newest first)
            if k in sst:
                return sst[k]
        return None

    def flush(self):
        if self.memtable:
            self.sstables.append(self.memtable)
            self.memtable = {}
            open(self.wal_path, "w").close()  # truncate WAL

with tempfile.TemporaryDirectory() as d:
    db = ToyLSM(os.path.join(d, "wal.log"))
    db.put("a", "1"); db.put("b", "2"); db.flush()
    db.put("a", "99")                         # updates in memtable
    print("get a =", db.get("a"))             # 99 (memtable beats SSTable)
    print("get b =", db.get("b"))             # 2  (flushed SSTable)

## Recap

- Schema-less: key/value are opaque bytes; store doesn't care.
- No version → **silent data loss** on concurrent writes.
- Versions + CAS → conflicts are **rejected and visible**; the client must be able to retry.
- LWW by timestamp → always available, but **silently discards** the loser, and clock skew
  means the "loser" may be the write that actually happened last.
- Vector clocks → preserve siblings, let the app merge.
- Real nodes store data in **LSM-trees** (WAL + memtable + SSTables) for fast writes.

Next notebook: deep dives into consistent hashing, quorums, and Merkle-tree anti-entropy.
